In [1]:
import pathlib

import pandas as pd
import swifter

from rdkit import Chem
from tqdm.auto import tqdm

In [2]:
NOTEBOOK_DIR = pathlib.Path().absolute()
DATA_DIR = NOTEBOOK_DIR.parents[1] / "data" / "solvation"
QUANTUM_GREEN_DIR = pathlib.Path("/home/shared/projects/quantum_green")
PAPER_DATA_DIR = QUANTUM_GREEN_DIR / "paper" / "data"

In [3]:
def canonical_smiles(smiles, isomeric=True, remove_atom_mapping=False):
    mol = Chem.MolFromSmiles(smiles)
    if remove_atom_mapping:
        for atom in mol.GetAtoms():
            atom.SetAtomMapNum(0)
    return Chem.MolToSmiles(mol, isomericSmiles=isomeric)


def first_letter(name):
    return next(s for s in name if s.isalpha())


pd.set_option('display.max_columns', None)


def head(df, n=2):
    display(df.head(n))
    print(f"Contains {len(df)} rows")
    
    
def swifter_apply(series, func, desc=None):
    if desc is None:
        desc = "Applying function"
    return series.swifter.progress_bar(True, desc).apply(func)

In [4]:
names_df = pd.read_csv(pathlib.Path.cwd() / "solvents.csv")
head(names_df)

,cosmo_name,smiles,inchi,cosmo_conf,source,exp_dielectric,T,source.1,polar,protic
0,"(1,1-dimethylethyl)benzene",CC(C)(C)c1ccccc1,"InChI=1/C10H14/c1-10(2,3)9-7-5-4-6-8-9/h4-8H,1...",1,COSMObase,2.359,20.0,NaN,0.0,0
1,"1-(1,1-dimethylethoxy)-2-propanol",CC(O)COC(C)(C)C,"InChI=1/C7H16O2/c1-6(8)5-9-7(2,3)4/h6,8H,5H2,1...",5,COSMObase,NaN,NaN,NaN,NaN,1


Contains 295 rows


In [5]:
non_isomeric_canonical_smiles = names_df["smiles"].apply(
    lambda x: canonical_smiles(x, isomeric=False)
)
(names_df["smiles"] != non_isomeric_canonical_smiles).sum()

np.int64(0)

In [6]:
smiles_to_name_mapping = names_df.set_index("smiles")["cosmo_name"].to_dict()

## Transition State Data

In [7]:
data = pd.read_csv(
    PAPER_DATA_DIR
    / "ts_solvation"
    / "FINAL_dG_solv_pruned_nov17_with_reactant_product_dGsolv.csv",
)
head(data)

,solvent_smiles,solute_name,solute_smiles,Gsolv (kcal/mol),Hsolv (kcal/mol),r1,r2,p1,p2,r1_Gsolv,r2_Gsolv,p1_Gsolv,p2_Gsolv,DDGsolv_forward (kcal/mol),DDGsolv_reverse (kcal/mol),r1_Hsolv,r2_Hsolv,p1_Hsolv,p2_Hsolv,DDHsolv_forward (kcal/mol),DDHsolv_reverse (kcal/mol)
0,CCCCOP(OCCCC)(OCCCC)=O,52940,[H:1][O:3][O:2].[H:4][N:25]1[C:19]([H:10])([H:...,-12.065855,-21.383296,[O:1][O:2][H:3],[C:1]([N:2]1[C:3]([H:13])([H:14])[C:4]2([H:15]...,[C:1]([N:2]1[C:3]([H:13])([H:14])[C:4]2([H:15]...,[O:1]([O:2][H:4])[H:3],-7.216970,-7.000390,-6.952042,-8.106657,2.151504,2.992844,-13.229345,-12.452517,-12.267439,-16.955998,4.298566,7.840141
1,CCCCCCCCCCCCC,52940,[H:1][O:3][O:2].[H:4][N:25]1[C:19]([H:10])([H:...,-7.794416,-14.052868,[O:1][O:2][H:3],[C:1]([N:2]1[C:3]([H:13])([H:14])[C:4]2([H:15]...,[C:1]([N:2]1[C:3]([H:13])([H:14])[C:4]2([H:15]...,[O:1]([O:2][H:4])[H:3],-0.905423,-6.425314,-6.485875,-1.134200,-0.463679,-0.174341,-3.678896,-11.593300,-11.621092,-4.325111,1.219328,1.893335


Contains 21684582 rows


In [8]:
unique_solvents = pd.DataFrame(
    data["solvent_smiles"].unique(), columns=["solvent_smiles"]
)
unique_solvents["non_isomeric_canonical_smiles"] = unique_solvents[
    "solvent_smiles"
].apply(lambda x: canonical_smiles(x, isomeric=False))
head(unique_solvents)

,solvent_smiles,non_isomeric_canonical_smiles
0,CCCCOP(OCCCC)(OCCCC)=O,CCCCOP(=O)(OCCCC)OCCCC
1,CCCCCCCCCCCCC,CCCCCCCCCCCCC


Contains 295 rows


In [9]:
(
    unique_solvents["solvent_smiles"]
    != unique_solvents["non_isomeric_canonical_smiles"]
).sum()

np.int64(85)

In [10]:
solvent_smiles_to_non_isomeric_canonical_smiles_mapping = unique_solvents.set_index(
    "solvent_smiles"
)["non_isomeric_canonical_smiles"].to_dict()

solvent_smiles_to_name_mapping = {
    k: smiles_to_name_mapping[v]
    for k, v in solvent_smiles_to_non_isomeric_canonical_smiles_mapping.items()
}

In [11]:
data["solvent_name"] = data["solvent_smiles"].map(solvent_smiles_to_name_mapping)
head(data)

,solvent_smiles,solute_name,solute_smiles,Gsolv (kcal/mol),Hsolv (kcal/mol),r1,r2,p1,p2,r1_Gsolv,r2_Gsolv,p1_Gsolv,p2_Gsolv,DDGsolv_forward (kcal/mol),DDGsolv_reverse (kcal/mol),r1_Hsolv,r2_Hsolv,p1_Hsolv,p2_Hsolv,DDHsolv_forward (kcal/mol),DDHsolv_reverse (kcal/mol),solvent_name
0,CCCCOP(OCCCC)(OCCCC)=O,52940,[H:1][O:3][O:2].[H:4][N:25]1[C:19]([H:10])([H:...,-12.065855,-21.383296,[O:1][O:2][H:3],[C:1]([N:2]1[C:3]([H:13])([H:14])[C:4]2([H:15]...,[C:1]([N:2]1[C:3]([H:13])([H:14])[C:4]2([H:15]...,[O:1]([O:2][H:4])[H:3],-7.216970,-7.000390,-6.952042,-8.106657,2.151504,2.992844,-13.229345,-12.452517,-12.267439,-16.955998,4.298566,7.840141,tri-n-butylphosphate
1,CCCCCCCCCCCCC,52940,[H:1][O:3][O:2].[H:4][N:25]1[C:19]([H:10])([H:...,-7.794416,-14.052868,[O:1][O:2][H:3],[C:1]([N:2]1[C:3]([H:13])([H:14])[C:4]2([H:15]...,[C:1]([N:2]1[C:3]([H:13])([H:14])[C:4]2([H:15]...,[O:1]([O:2][H:4])[H:3],-0.905423,-6.425314,-6.485875,-1.134200,-0.463679,-0.174341,-3.678896,-11.593300,-11.621092,-4.325111,1.219328,1.893335,tridecane


Contains 21684582 rows


In [12]:
unique_reactants = pd.DataFrame(
    set().union(*[data[col] for col in ["r1", "r2", "p1", "p2"]]),
    columns=["atom_mapped_smiles"],
)
head(unique_reactants)

,atom_mapped_smiles
0,[C:1]([C:2]([N:3]([C:4]([C:5]([O:6][H:23])([H:...
1,[C:1]([C:2]([C:3]([C:4]1=[N:5][N:6][C:7]([C:8]...


Contains 133253 rows


In [13]:
unique_reactants["canonical_smiles"] = swifter_apply(
    unique_reactants["atom_mapped_smiles"],
    lambda x: canonical_smiles(x, remove_atom_mapping=True),
    "Generating canonical smiles",
)
head(unique_reactants)

Generating canonical smiles:   0%|          | 0/133253 [00:00<?, ?it/s]

,atom_mapped_smiles,canonical_smiles
0,[C:1]([C:2]([N:3]([C:4]([C:5]([O:6][H:23])([H:...,CCNC(CO)C(CC)C(N)CC
1,[C:1]([C:2]([C:3]([C:4]1=[N:5][N:6][C:7]([C:8]...,CCCC1=N[N]C(C)=N1


Contains 133253 rows


In [14]:
species_to_canonical_smiles_mapping = unique_reactants.set_index(
    "atom_mapped_smiles"
)["canonical_smiles"].to_dict()

In [15]:
for col in ["r1", "r2", "p1", "p2"]:
    data[col + "_smiles"] = swifter_apply(
        data[col],
        lambda x: species_to_canonical_smiles_mapping.get(x),
        f"Mapping {col}",
    )
head(data)

Mapping r1:   0%|          | 0/21684582 [00:00<?, ?it/s]

Mapping r2:   0%|          | 0/21684582 [00:00<?, ?it/s]

Mapping p1:   0%|          | 0/21684582 [00:00<?, ?it/s]

Mapping p2:   0%|          | 0/21684582 [00:00<?, ?it/s]

,solvent_smiles,solute_name,solute_smiles,Gsolv (kcal/mol),Hsolv (kcal/mol),r1,r2,p1,p2,r1_Gsolv,r2_Gsolv,p1_Gsolv,p2_Gsolv,DDGsolv_forward (kcal/mol),DDGsolv_reverse (kcal/mol),r1_Hsolv,r2_Hsolv,p1_Hsolv,p2_Hsolv,DDHsolv_forward (kcal/mol),DDHsolv_reverse (kcal/mol),solvent_name,r1_smiles,r2_smiles,p1_smiles,p2_smiles
0,CCCCOP(OCCCC)(OCCCC)=O,52940,[H:1][O:3][O:2].[H:4][N:25]1[C:19]([H:10])([H:...,-12.065855,-21.383296,[O:1][O:2][H:3],[C:1]([N:2]1[C:3]([H:13])([H:14])[C:4]2([H:15]...,[C:1]([N:2]1[C:3]([H:13])([H:14])[C:4]2([H:15]...,[O:1]([O:2][H:4])[H:3],-7.216970,-7.000390,-6.952042,-8.106657,2.151504,2.992844,-13.229345,-12.452517,-12.267439,-16.955998,4.298566,7.840141,tri-n-butylphosphate,[O]O,CN1CC2CNCC2C1,CN1CC2C[N]CC2C1,OO
1,CCCCCCCCCCCCC,52940,[H:1][O:3][O:2].[H:4][N:25]1[C:19]([H:10])([H:...,-7.794416,-14.052868,[O:1][O:2][H:3],[C:1]([N:2]1[C:3]([H:13])([H:14])[C:4]2([H:15]...,[C:1]([N:2]1[C:3]([H:13])([H:14])[C:4]2([H:15]...,[O:1]([O:2][H:4])[H:3],-0.905423,-6.425314,-6.485875,-1.134200,-0.463679,-0.174341,-3.678896,-11.593300,-11.621092,-4.325111,1.219328,1.893335,tridecane,[O]O,CN1CC2CNCC2C1,CN1CC2C[N]CC2C1,OO


Contains 21684582 rows


In [16]:
data["rxn_smiles"] = (
    data["r1_smiles"]
    + "."
    + data["r2_smiles"]
    + ">>"
    + data["p1_smiles"]
    + "."
    + data["p2_smiles"]
)
head(data)

,solvent_smiles,solute_name,solute_smiles,Gsolv (kcal/mol),Hsolv (kcal/mol),r1,r2,p1,p2,r1_Gsolv,r2_Gsolv,p1_Gsolv,p2_Gsolv,DDGsolv_forward (kcal/mol),DDGsolv_reverse (kcal/mol),r1_Hsolv,r2_Hsolv,p1_Hsolv,p2_Hsolv,DDHsolv_forward (kcal/mol),DDHsolv_reverse (kcal/mol),solvent_name,r1_smiles,r2_smiles,p1_smiles,p2_smiles,rxn_smiles
0,CCCCOP(OCCCC)(OCCCC)=O,52940,[H:1][O:3][O:2].[H:4][N:25]1[C:19]([H:10])([H:...,-12.065855,-21.383296,[O:1][O:2][H:3],[C:1]([N:2]1[C:3]([H:13])([H:14])[C:4]2([H:15]...,[C:1]([N:2]1[C:3]([H:13])([H:14])[C:4]2([H:15]...,[O:1]([O:2][H:4])[H:3],-7.216970,-7.000390,-6.952042,-8.106657,2.151504,2.992844,-13.229345,-12.452517,-12.267439,-16.955998,4.298566,7.840141,tri-n-butylphosphate,[O]O,CN1CC2CNCC2C1,CN1CC2C[N]CC2C1,OO,[O]O.CN1CC2CNCC2C1>>CN1CC2C[N]CC2C1.OO
1,CCCCCCCCCCCCC,52940,[H:1][O:3][O:2].[H:4][N:25]1[C:19]([H:10])([H:...,-7.794416,-14.052868,[O:1][O:2][H:3],[C:1]([N:2]1[C:3]([H:13])([H:14])[C:4]2([H:15]...,[C:1]([N:2]1[C:3]([H:13])([H:14])[C:4]2([H:15]...,[O:1]([O:2][H:4])[H:3],-0.905423,-6.425314,-6.485875,-1.134200,-0.463679,-0.174341,-3.678896,-11.593300,-11.621092,-4.325111,1.219328,1.893335,tridecane,[O]O,CN1CC2CNCC2C1,CN1CC2C[N]CC2C1,OO,[O]O.CN1CC2CNCC2C1>>CN1CC2C[N]CC2C1.OO


Contains 21684582 rows


In [17]:
ouput_columns = [
    "rxn_smiles",
    "Gsolv (kcal/mol)",
    "r1_Gsolv",
    "r2_Gsolv",
    "p1_Gsolv",
    "p2_Gsolv",
    "DDGsolv_forward (kcal/mol)",
    "DDGsolv_reverse (kcal/mol)",
    "Hsolv (kcal/mol)",
    "r1_Hsolv",
    "r2_Hsolv",
    "p1_Hsolv",
    "p2_Hsolv",
    "DDHsolv_forward (kcal/mol)",
    "DDHsolv_reverse (kcal/mol)",
]

In [18]:
grouped_solvents = data.groupby("solvent_name")

In [19]:
DIR = DATA_DIR / "transition_states"
DIR.mkdir(parents=True, exist_ok=True)
for name, group in tqdm(grouped_solvents, total=len(unique_solvents)):
    subdir = DIR / first_letter(name)
    subdir.mkdir(parents=True, exist_ok=True)
    group[ouput_columns].to_csv(
        subdir / f"{name}.csv", index=False, float_format="%.10g"
    )

  0%|          | 0/295 [00:00<?, ?it/s]